In [1]:
# =============================================================================
# Time series with an RNN (LSTM) — Google stock opening prices
# =============================================================================
# Goal: learn patterns from past prices so the network can predict the *next*
# opening price (in scaled units). This is a classic "many past steps → one
# future value" setup used in tutorials; real trading needs much more care.
#
# Libraries:
#   numpy  — fast arrays (later we stack windows into tensors).
#   pandas — load CSV and inspect rows/columns.
#   matplotlib — plots (e.g. predicted vs actual) in later notebook sections.
#   sklearn.preprocessing.MinMaxScaler — put prices in [0, 1] so training is stable.
# In Jupyter / VS Code, select this folder's `.venv` kernel so NumPy matches TensorFlow.
#
# TensorFlow 2.13 needs NumPy 1.24.x. If pip ever upgrades you to NumPy 2.x, fix the
# install and restart the kernel (imports cannot switch NumPy major in the same session).
import importlib.metadata as _im
import subprocess
import sys

def _pip(*packages):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        stdout=sys.stderr,
    )


try:
    _np_ver = _im.version("numpy")
    if int(_np_ver.split(".")[0]) >= 2:
        _pip("numpy==1.24.3")
        raise RuntimeError(
            "NumPy was set to 1.24.3 for TensorFlow. Use Kernel → Restart, then Run All from this cell."
        )
except _im.PackageNotFoundError:
    # Fresh env or wrong kernel: install the stack this notebook needs (then continue).
    _pip("numpy==1.24.3", "pandas", "matplotlib", "scikit-learn")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

dataset_train = pd.read_csv("Google_Stock_Price_Train.csv")

# Use one numeric column as the target series. iloc[:, 1:2] is "Open" and keeps
# shape (n_rows, 1); a 2D column vector plays nicely with MinMaxScaler and Keras.
training_set = dataset_train.iloc[:, 1:2].values

In [2]:
# Peek at the table: column names, missing values (none expected), and scale of numbers.
# head(10) shows the first 10 rows only so the notebook stays readable.
dataset_train.head(10)

,Date,Open,High,Low,Close,Volume
0,1/3/2012,325.25,332.83,324.97,663.59,"7,380,500"
1,1/4/2012,331.27,333.87,329.08,666.45,"5,749,400"
2,1/5/2012,329.83,330.75,326.89,657.21,"6,590,300"
3,1/6/2012,328.34,328.77,323.68,648.24,"5,405,900"
4,1/9/2012,322.04,322.29,309.46,620.76,"11,688,800"
5,1/10/2012,313.70,315.72,307.30,621.43,"8,824,000"
6,1/11/2012,310.59,313.52,309.40,624.25,"4,817,800"
7,1/12/2012,314.43,315.26,312.08,627.92,"3,764,400"
8,1/13/2012,311.96,312.30,309.37,623.28,"4,631,800"
9,1/17/2012,314.81,314.81,311.67,626.86,"3,832,800"


In [3]:
# Scale to [0, 1]: neural nets like inputs in a modest, fixed range.
# fit_transform learns min/max *from this training column* and transforms it.
# Rule for real projects: fit the scaler on training data only, then transform
# test data with the same scaler (so test numbers stay comparable to training).
sc = MinMaxScaler(feature_range=(0, 1))
training_set_scaled = sc.fit_transform(training_set)


In [4]:
# Build supervised learning examples from a single long price series.
# For each index i, x = the previous `time_steps` prices, y = the price at day i.
# So each y is "one step ahead" of the last value in the matching x window.
#
# time_steps = 60 is a hyperparameter: more days → richer context but fewer
# training windows (the loop starts at index 60, not 0).

time_steps = 60
x_train = []
y_train = []

for i in range(time_steps, len(training_set_scaled)):
    x_train.append(training_set_scaled[i - time_steps : i, 0])
    y_train.append(training_set_scaled[i, 0])

x_train = np.array(x_train)
y_train = np.array(y_train)


In [5]:
# LSTM layers in Keras expect 3D input: (number_of_windows, timesteps, features).
# We have one feature per day (open price), so the last dimension is 1.
x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))

In [6]:
# =============================================================================
# Part 2 — stacked LSTM model (TensorFlow / Keras)
# =============================================================================
# LSTM (Long Short-Term Memory): a recurrent layer that carries hidden state through
# time so the network can use order in the sequence (unlike a plain feed-forward net
# that would see one long flat vector without a clear time axis).
#
# return_sequences=True  → output one vector per timestep (needed when another LSTM follows).
# return_sequences=False → output only the last timestep (typical before a Dense head).
#
# If NumPy and TensorFlow were built against different ABIs, import fails with a
# "numpy.dtype size changed" ValueError — reinstall pins, then restart the kernel.
import subprocess
import sys

try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, LSTM, Dropout
except ValueError as e:
    if "numpy.dtype" not in str(e) and "incompatibility" not in str(e):
        raise
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--force-reinstall",
            "numpy==1.24.3",
            "tensorflow-macos==2.13.0",
            "tensorflow-metal==1.0.1",
        ],
        stdout=sys.stderr,
    )
    raise RuntimeError(
        "Reinstalled NumPy + TensorFlow (binary mismatch fix). "
        "Kernel → Restart, then Run All from cell 0."
    ) from e

regressor = Sequential()

# Dropout: during training, randomly drop some connections (rate=0.2 → 20% zeroed).
# That reduces "memorizing" noise; it is a different idea from L2 weight decay but
# both encourage simpler, more general patterns.

regressor.add(
    LSTM(units=50, return_sequences=True, input_shape=(x_train.shape[1], 1))
)
regressor.add(Dropout(0.2))

regressor.add(LSTM(units=50, return_sequences=True))
regressor.add(Dropout(0.2))

regressor.add(LSTM(units=50, return_sequences=True))
regressor.add(Dropout(0.2))

regressor.add(LSTM(units=50))  # last LSTM: default return_sequences=False
regressor.add(Dropout(0.2))

regressor.add(Dense(units=1))  # one continuous prediction (scaled open price)

# ----- Loss -----
# mean_squared_error (MSE): average squared gap between prediction and target.
# Squaring penalizes big errors more; common for regression.

# ----- Optimizer: Adam (and how it compares, in plain language) -----
# Adam = "Adaptive Moment Estimation". For each weight, it keeps two moving averages:
#   (1) the gradient — like momentum: "remember velocity downhill, smooth zig-zags";
#   (2) the squared gradient — like RMSprop: rescale steps when gradients are noisy.
# It then takes a step that adapts per-parameter and usually works well with default
# hyperparameters — a good first choice for many sequence models and this tutorial.
#
# Other optimizers you will see in Keras (same loss, different update rule):
#   SGD        — Stochastic Gradient Descent: step along -gradient. Very simple; often
#                needs you to tune learning rate (and maybe momentum) carefully.
#   RMSprop    — Scales each parameter by a running average of squared gradients; was
#                popular for RNNs before Adam became the default "easy" option.
#   Adagrad    — Shrinks the learning rate more for parameters that change a lot; can
#                help sparse problems but sometimes slows learning too much later on.
#   AdamW      — Like Adam, but fixes how weight decay is applied; often used in newer
#                large models when people tune weight decay seriously.
#
# Here we use the string "adam" (Keras default hyperparameters). You could instead write
# optimizer=tf.keras.optimizers.Adam(learning_rate=0.001) to experiment with step size.
regressor.compile(optimizer="adam", loss="mean_squared_error")


/Users/payam/RNN PRACITICNG /.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
2026-04-20 03:14:41.683189: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5
2026-04-20 03:14:41.683209: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2026-04-20 03:14:41.683213: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 12.48 GB
2026-04-20 03:14:41.683235: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-20 03:14:41.683247: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 wit

In [7]:
# fit(...) runs backpropagation: for each batch, compare predictions to y_train,
# compute MSE, adjust weights with Adam. One epoch = one full pass over all windows.
#
# batch_size: how many (x, y) pairs per gradient update — larger batches → smoother
#   gradients but more memory; 32 is a common default.
# epochs: increase once the pipeline works (watch loss on train vs validation if you split).
# verbose=1 prints a progress bar with loss each epoch.
#
# EPOCHS: 10 finishes in ~1 minute on a laptop; set to 100 only when you want a long run.

EPOCHS = 10
regressor.fit(x_train, y_train, epochs=EPOCHS, batch_size=32, verbose=1)


Epoch 1/10


2026-04-20 03:14:42.963286: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2026-04-20 03:14:43.140018: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2026-04-20 03:14:43.190345: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2026-04-20 03:14:43.242019: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2026-04-20 03:14:43.288071: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2026-04-20 03:14:43.380414: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2026-04-20 03:14:43.459473: I tensorflow/core/grappler/optimizers/cust

38/38 [==============================] - 2s 21ms/step - loss: 0.0450
Epoch 2/10
38/38 [==============================] - 1s 20ms/step - loss: 0.0046
Epoch 3/10
38/38 [==============================] - 1s 18ms/step - loss: 0.0031
Epoch 4/10
38/38 [==============================] - 1s 17ms/step - loss: 0.0027
Epoch 5/10
38/38 [==============================] - 1s 17ms/step - loss: 0.0023
Epoch 6/10
38/38 [==============================] - 1s 18ms/step - loss: 0.0023
Epoch 7/10
38/38 [==============================] - 1s 18ms/step - loss: 0.0023
Epoch 8/10
38/38 [==============================] - 1s 18ms/step - loss: 0.0021
Epoch 9/10
38/38 [==============================] - 1s 19ms/step - loss: 0.0021
Epoch 10/10
38/38 [==============================] - 1s 17ms/step - loss: 0.0021


In [ ]:
# Part - Making the predictions and visualising the results
# Getting the real stock price of 2017
dataset_test = pd.read_csv("Google_Stock_Price_Test.csv")
real_stock_price = dataset_test.iloc[:, 1:2].values

# Combine train + test 'Open' columns to build test windows that include
# the 60 days of context that precede the first test day.
dataset_total = pd.concat((dataset_train['Open'], dataset_test['Open']), axis=0)
inputs = dataset_total[len(dataset_total) - len(dataset_test) - 60:].values
inputs = inputs.reshape(-1, 1)   # fix: reshap → reshape
inputs = sc.transform(inputs)

x_test = []
for i in range(60, 60 + len(dataset_test)):   # fix: range(60,80) → dynamic length
    x_test.append(inputs[i - 60:i, 0])

x_test = np.array(x_test)                                          # fix: X_test → x_test
x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], 1))  # fix: sahpe → shape

predicted_stock_price = regressor.predict(x_test)                   # fix: X_TEST → x_test
predicted_stock_price = sc.inverse_transform(predicted_stock_price)  # fix: variable name

# Visualising the results
plt.plot(real_stock_price, color='red', label='Real Google Stock Price')
plt.plot(predicted_stock_price, color='blue', label='Predicted Google Stock Price')
plt.title('Google Stock Price Prediction')
plt.xlabel('Time')
plt.ylabel('Google Stock Price')
plt.legend()
plt.show()
